In [1]:
import cv2
import mediapipe as mp
import numpy as np
import os


In [2]:
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True)

print("Pose model initialized")


Pose model initialized


In [3]:
def calculate_angle(a, b, c):
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    ba = a - b
    bc = c - b

    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    angle = np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))

    return angle


In [4]:
pose1_angles = {
    "left_elbow": [],
    "right_elbow": [],
    "left_knee": [],
    "right_knee": [],
    "left_shoulder": [],
    "right_shoulder": []
}


In [5]:
folder_path = "POSE1"

for img_name in os.listdir(folder_path):
    img_path = os.path.join(folder_path, img_name)
    image = cv2.imread(img_path)

    if image is None:
        continue

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = pose.process(image_rgb)

    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark

        # Elbows
        left_elbow = calculate_angle(
            [lm[mp_pose.PoseLandmark.LEFT_SHOULDER].x,
             lm[mp_pose.PoseLandmark.LEFT_SHOULDER].y],
            [lm[mp_pose.PoseLandmark.LEFT_ELBOW].x,
             lm[mp_pose.PoseLandmark.LEFT_ELBOW].y],
            [lm[mp_pose.PoseLandmark.LEFT_WRIST].x,
             lm[mp_pose.PoseLandmark.LEFT_WRIST].y]
        )

        right_elbow = calculate_angle(
            [lm[mp_pose.PoseLandmark.RIGHT_SHOULDER].x,
             lm[mp_pose.PoseLandmark.RIGHT_SHOULDER].y],
            [lm[mp_pose.PoseLandmark.RIGHT_ELBOW].x,
             lm[mp_pose.PoseLandmark.RIGHT_ELBOW].y],
            [lm[mp_pose.PoseLandmark.RIGHT_WRIST].x,
             lm[mp_pose.PoseLandmark.RIGHT_WRIST].y]
        )

        # Knees
        left_knee = calculate_angle(
            [lm[mp_pose.PoseLandmark.LEFT_HIP].x,
             lm[mp_pose.PoseLandmark.LEFT_HIP].y],
            [lm[mp_pose.PoseLandmark.LEFT_KNEE].x,
             lm[mp_pose.PoseLandmark.LEFT_KNEE].y],
            [lm[mp_pose.PoseLandmark.LEFT_ANKLE].x,
             lm[mp_pose.PoseLandmark.LEFT_ANKLE].y]
        )

        right_knee = calculate_angle(
            [lm[mp_pose.PoseLandmark.RIGHT_HIP].x,
             lm[mp_pose.PoseLandmark.RIGHT_HIP].y],
            [lm[mp_pose.PoseLandmark.RIGHT_KNEE].x,
             lm[mp_pose.PoseLandmark.RIGHT_KNEE].y],
            [lm[mp_pose.PoseLandmark.RIGHT_ANKLE].x,
             lm[mp_pose.PoseLandmark.RIGHT_ANKLE].y]
        )

        # Shoulders (IMPORTANT)
        left_shoulder = calculate_angle(
            [lm[mp_pose.PoseLandmark.LEFT_ELBOW].x,
             lm[mp_pose.PoseLandmark.LEFT_ELBOW].y],
            [lm[mp_pose.PoseLandmark.LEFT_SHOULDER].x,
             lm[mp_pose.PoseLandmark.LEFT_SHOULDER].y],
            [lm[mp_pose.PoseLandmark.LEFT_HIP].x,
             lm[mp_pose.PoseLandmark.LEFT_HIP].y]
        )

        right_shoulder = calculate_angle(
            [lm[mp_pose.PoseLandmark.RIGHT_ELBOW].x,
             lm[mp_pose.PoseLandmark.RIGHT_ELBOW].y],
            [lm[mp_pose.PoseLandmark.RIGHT_SHOULDER].x,
             lm[mp_pose.PoseLandmark.RIGHT_SHOULDER].y],
            [lm[mp_pose.PoseLandmark.RIGHT_HIP].x,
             lm[mp_pose.PoseLandmark.RIGHT_HIP].y]
        )

        # Store values
        pose1_angles["left_elbow"].append(left_elbow)
        pose1_angles["right_elbow"].append(right_elbow)
        pose1_angles["left_knee"].append(left_knee)
        pose1_angles["right_knee"].append(right_knee)
        pose1_angles["left_shoulder"].append(left_shoulder)
        pose1_angles["right_shoulder"].append(right_shoulder)

        print(f"\nImage: {img_name}")
        print("Left Elbow:", int(left_elbow))
        print("Right Elbow:", int(right_elbow))
        print("Left Knee:", int(left_knee))
        print("Right Knee:", int(right_knee))
        print("Left Shoulder:", int(left_shoulder))
        print("Right Shoulder:", int(right_shoulder))



Image: p11.jpg
Left Elbow: 49
Right Elbow: 46
Left Knee: 179
Right Knee: 176
Left Shoulder: 36
Right Shoulder: 38

Image: p12.jpg
Left Elbow: 59
Right Elbow: 60
Left Knee: 179
Right Knee: 176
Left Shoulder: 38
Right Shoulder: 40

Image: p13.jpg
Left Elbow: 33
Right Elbow: 36
Left Knee: 179
Right Knee: 174
Left Shoulder: 45
Right Shoulder: 43


In [6]:
avg_pose1 = {k: int(np.mean(v)) for k, v in pose1_angles.items()}

avg_pose1


{'left_elbow': 47,
 'right_elbow': 47,
 'left_knee': 179,
 'right_knee': 176,
 'left_shoulder': 40,
 'right_shoulder': 41}

In [7]:
import json

pose1_template = {
    "pose_name": "Pranamasana",
    "angles": {
        "left_elbow": [avg_pose1["left_elbow"] - 15, avg_pose1["left_elbow"] + 15],
        "right_elbow": [avg_pose1["right_elbow"] - 15, avg_pose1["right_elbow"] + 15],
        "left_knee": [170, 180],
        "right_knee": [170, 180],
        "left_shoulder": [avg_pose1["left_shoulder"] - 15, avg_pose1["left_shoulder"] + 15],
        "right_shoulder": [avg_pose1["right_shoulder"] - 15, avg_pose1["right_shoulder"] + 15]
    }
}

with open("pose1_pranamasana.json", "w") as f:
    json.dump(pose1_template, f, indent=4)

print("Pose-1 JSON saved successfully")


Pose-1 JSON saved successfully


In [8]:
with open("pose1_pranamasana.json", "r") as f:
    pose1_template = json.load(f)

pose1_ranges = pose1_template["angles"]


In [9]:
import time  

In [10]:
''' cap = cv2.VideoCapture(0)

SESSION_TIME = 180  # 3 minutes
session_start = time.time()

pose_start = None
total_pose_time = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    current_time = time.time()
    session_elapsed = current_time - session_start

    # Stop camera after 3 minutes
    if session_elapsed >= SESSION_TIME:
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(frame_rgb)

    feedback = []
    pose_correct = True

    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark

        left_elbow = calculate_angle(
            [lm[11].x, lm[11].y],
            [lm[13].x, lm[13].y],
            [lm[15].x, lm[15].y]
        )

        right_elbow = calculate_angle(
            [lm[12].x, lm[12].y],
            [lm[14].x, lm[14].y],
            [lm[16].x, lm[16].y]
        )

        left_knee = calculate_angle(
            [lm[23].x, lm[23].y],
            [lm[25].x, lm[25].y],
            [lm[27].x, lm[27].y]
        )

        right_knee = calculate_angle(
            [lm[24].x, lm[24].y],
            [lm[26].x, lm[26].y],
            [lm[28].x, lm[28].y]
        )

        def check(angle, min_val, max_val, msg):
            if not (min_val <= angle <= max_val):
                feedback.append(msg)
                return False
            return True

        pose_correct &= check(left_elbow, *pose1_ranges["left_elbow"], "Adjust left elbow")
        pose_correct &= check(right_elbow, *pose1_ranges["right_elbow"], "Adjust right elbow")
        pose_correct &= check(left_knee, *pose1_ranges["left_knee"], "Straighten left knee")
        pose_correct &= check(right_knee, *pose1_ranges["right_knee"], "Straighten right knee")

    # --------- CUMULATIVE TIMER LOGIC ----------
    if pose_correct and results.pose_landmarks:
        if pose_start is None:
            pose_start = current_time
    else:
        if pose_start is not None:
            total_pose_time += current_time - pose_start
            pose_start = None

    # --------- DISPLAY ----------
    cv2.putText(frame, f"Session: {int(session_elapsed)} / 180 sec",
                (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

    cv2.putText(frame, f"Correct Pose Time: {int(total_pose_time)} sec",
                (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

    y = 120
    for msg in feedback:
        cv2.putText(frame, msg, (20, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)
        y += 30

    cv2.imshow("Pranamasana Live Feedback", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Add last segment if pose was correct when session ended
if pose_start is not None:
    total_pose_time += time.time() - pose_start

print("SESSION COMPLETED")
print(f"Total Correct Pose Time: {int(total_pose_time)} seconds")

cap.release()
cv2.destroyAllWindows() **?'''


' cap = cv2.VideoCapture(0)\n\nSESSION_TIME = 180  # 3 minutes\nsession_start = time.time()\n\npose_start = None\ntotal_pose_time = 0\n\nwhile cap.isOpened():\n    ret, frame = cap.read()\n    if not ret:\n        break\n\n    current_time = time.time()\n    session_elapsed = current_time - session_start\n\n    # Stop camera after 3 minutes\n    if session_elapsed >= SESSION_TIME:\n        break\n\n    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)\n    results = pose.process(frame_rgb)\n\n    feedback = []\n    pose_correct = True\n\n    if results.pose_landmarks:\n        lm = results.pose_landmarks.landmark\n\n        left_elbow = calculate_angle(\n            [lm[11].x, lm[11].y],\n            [lm[13].x, lm[13].y],\n            [lm[15].x, lm[15].y]\n        )\n\n        right_elbow = calculate_angle(\n            [lm[12].x, lm[12].y],\n            [lm[14].x, lm[14].y],\n            [lm[16].x, lm[16].y]\n        )\n\n        left_knee = calculate_angle(\n            [lm[23].x, l

In [11]:
POSE2_FOLDER = "POSE2"

# Store raw values
angles_data = {
    "left_elbow": [],
    "right_elbow": [],
    "left_hip": [],
    "right_hip": []
}

for img_name in os.listdir(POSE2_FOLDER):
    img_path = os.path.join(POSE2_FOLDER, img_name)

    image = cv2.imread(img_path)
    if image is None:
        continue

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = pose.process(image_rgb)

    if not results.pose_landmarks:
        continue

    lm = results.pose_landmarks.landmark

    # ----- ELBOW ANGLES -----
    left_elbow = calculate_angle(
        [lm[11].x, lm[11].y],
        [lm[13].x, lm[13].y],
        [lm[15].x, lm[15].y]
    )

    right_elbow = calculate_angle(
        [lm[12].x, lm[12].y],
        [lm[14].x, lm[14].y],
        [lm[16].x, lm[16].y]
    )

    # ----- HIP ANGLES (BACK BEND) -----
    left_hip = calculate_angle(
        [lm[11].x, lm[11].y],
        [lm[23].x, lm[23].y],
        [lm[25].x, lm[25].y]
    )

    right_hip = calculate_angle(
        [lm[12].x, lm[12].y],
        [lm[24].x, lm[24].y],
        [lm[26].x, lm[26].y]
    )

    angles_data["left_elbow"].append(left_elbow)
    angles_data["right_elbow"].append(right_elbow)
    angles_data["left_hip"].append(left_hip)
    angles_data["right_hip"].append(right_hip)

# ---- AVERAGE VALUES ----
avg_angles = {
    key: round(float(np.mean(values)), 2)
    for key, values in angles_data.items()
    if values
}

# ---- FINAL JSON ----
asana2_json = {
    "asana": "hasta_uttanasana",
    "angles_avg": avg_angles
}

with open("asana2_hasta_uttanasana_avg.json", "w") as f:
    json.dump(asana2_json, f, indent=4)

print("Asana-2 average angles saved successfully ✅")
print(asana2_json)

Asana-2 average angles saved successfully ✅
{'asana': 'hasta_uttanasana', 'angles_avg': {'left_elbow': 151.43, 'right_elbow': 145.2, 'left_hip': 132.53, 'right_hip': 133.48}}


In [12]:


folder_path = "POSE3"

pose3_angles = {
    "left_elbow": [],
    "right_elbow": [],
    "left_knee": [],
    "right_knee": [],
    "left_hip": [],
    "right_hip": []
}

for img_name in os.listdir(folder_path):
    img_path = os.path.join(folder_path, img_name)
    image = cv2.imread(img_path)

    if image is None:
        continue

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = pose.process(image_rgb)

    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark

        # -------- ELBOWS --------
        left_elbow = calculate_angle(
            [lm[mp_pose.PoseLandmark.LEFT_SHOULDER].x,
             lm[mp_pose.PoseLandmark.LEFT_SHOULDER].y],
            [lm[mp_pose.PoseLandmark.LEFT_ELBOW].x,
             lm[mp_pose.PoseLandmark.LEFT_ELBOW].y],
            [lm[mp_pose.PoseLandmark.LEFT_WRIST].x,
             lm[mp_pose.PoseLandmark.LEFT_WRIST].y]
        )

        right_elbow = calculate_angle(
            [lm[mp_pose.PoseLandmark.RIGHT_SHOULDER].x,
             lm[mp_pose.PoseLandmark.RIGHT_SHOULDER].y],
            [lm[mp_pose.PoseLandmark.RIGHT_ELBOW].x,
             lm[mp_pose.PoseLandmark.RIGHT_ELBOW].y],
            [lm[mp_pose.PoseLandmark.RIGHT_WRIST].x,
             lm[mp_pose.PoseLandmark.RIGHT_WRIST].y]
        )

        # -------- KNEES --------
        left_knee = calculate_angle(
            [lm[mp_pose.PoseLandmark.LEFT_HIP].x,
             lm[mp_pose.PoseLandmark.LEFT_HIP].y],
            [lm[mp_pose.PoseLandmark.LEFT_KNEE].x,
             lm[mp_pose.PoseLandmark.LEFT_KNEE].y],
            [lm[mp_pose.PoseLandmark.LEFT_ANKLE].x,
             lm[mp_pose.PoseLandmark.LEFT_ANKLE].y]
        )

        right_knee = calculate_angle(
            [lm[mp_pose.PoseLandmark.RIGHT_HIP].x,
             lm[mp_pose.PoseLandmark.RIGHT_HIP].y],
            [lm[mp_pose.PoseLandmark.RIGHT_KNEE].x,
             lm[mp_pose.PoseLandmark.RIGHT_KNEE].y],
            [lm[mp_pose.PoseLandmark.RIGHT_ANKLE].x,
             lm[mp_pose.PoseLandmark.RIGHT_ANKLE].y]
        )

        # -------- HIPS (FORWARD BEND) --------
        left_hip = calculate_angle(
            [lm[mp_pose.PoseLandmark.LEFT_SHOULDER].x,
             lm[mp_pose.PoseLandmark.LEFT_SHOULDER].y],
            [lm[mp_pose.PoseLandmark.LEFT_HIP].x,
             lm[mp_pose.PoseLandmark.LEFT_HIP].y],
            [lm[mp_pose.PoseLandmark.LEFT_KNEE].x,
             lm[mp_pose.PoseLandmark.LEFT_KNEE].y]
        )

        right_hip = calculate_angle(
            [lm[mp_pose.PoseLandmark.RIGHT_SHOULDER].x,
             lm[mp_pose.PoseLandmark.RIGHT_SHOULDER].y],
            [lm[mp_pose.PoseLandmark.RIGHT_HIP].x,
             lm[mp_pose.PoseLandmark.RIGHT_HIP].y],
            [lm[mp_pose.PoseLandmark.RIGHT_KNEE].x,
             lm[mp_pose.PoseLandmark.RIGHT_KNEE].y]
        )

        pose3_angles["left_elbow"].append(left_elbow)
        pose3_angles["right_elbow"].append(right_elbow)
        pose3_angles["left_knee"].append(left_knee)
        pose3_angles["right_knee"].append(right_knee)
        pose3_angles["left_hip"].append(left_hip)
        pose3_angles["right_hip"].append(right_hip)

        print(f"\nImage: {img_name}")
        print("Left Elbow:", int(left_elbow))
        print("Right Elbow:", int(right_elbow))
        print("Left Knee:", int(left_knee))
        print("Right Knee:", int(right_knee))
        print("Left Hip:", int(left_hip))
        print("Right Hip:", int(right_hip))

# -------- STATS (MIN / MAX / AVG) --------
pose3_stats = {}
for k, v in pose3_angles.items():
    pose3_stats[k] = {
        "min": int(np.min(v)),
        "max": int(np.max(v)),
        "avg": int(np.mean(v))
    }

print("\nPOSE-3 ANGLE STATS:")
for k, s in pose3_stats.items():
    print(f"{k} → min:{s['min']}  max:{s['max']}  avg:{s['avg']}")

# -------- JSON TEMPLATE --------
pose3_template = {
    "pose_name": "Padahastasana",
    "angles": {
        "left_elbow": [pose3_stats["left_elbow"]["min"], pose3_stats["left_elbow"]["max"]],
        "right_elbow": [pose3_stats["right_elbow"]["min"], pose3_stats["right_elbow"]["max"]],
        "left_knee": [170, 180],
        "right_knee": [170, 180],
        "left_hip": [pose3_stats["left_hip"]["min"], pose3_stats["left_hip"]["max"]],
        "right_hip": [pose3_stats["right_hip"]["min"], pose3_stats["right_hip"]["max"]]
    }
}

with open("pose3_padahastasana.json", "w") as f:
    json.dump(pose3_template, f, indent=4)

print("\nPose-3 JSON saved successfully ✅")



Image: 31.jpg
Left Elbow: 174
Right Elbow: 151
Left Knee: 179
Right Knee: 177
Left Hip: 10
Right Hip: 16

Image: 32.jpg
Left Elbow: 179
Right Elbow: 173
Left Knee: 170
Right Knee: 170
Left Hip: 2
Right Hip: 3

Image: 33.jpg
Left Elbow: 107
Right Elbow: 93
Left Knee: 173
Right Knee: 174
Left Hip: 8
Right Hip: 7

POSE-3 ANGLE STATS:
left_elbow → min:107  max:179  avg:153
right_elbow → min:93  max:173  avg:139
left_knee → min:170  max:179  avg:174
right_knee → min:170  max:177  avg:174
left_hip → min:2  max:10  avg:7
right_hip → min:3  max:16  avg:9

Pose-3 JSON saved successfully ✅


In [13]:
folder_path = "POSE4"

pose4_angles = {
    "front_knee": [],
    "back_knee": [],
    "left_hip": [],
    "right_hip": [],
    "left_elbow": [],
    "right_elbow": []
}

for img_name in os.listdir(folder_path):
    img_path = os.path.join(folder_path, img_name)
    image = cv2.imread(img_path)

    if image is None:
        continue

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = pose.process(image_rgb)

    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark

        # ---- KNEES ----
        left_knee = calculate_angle(
            [lm[mp_pose.PoseLandmark.LEFT_HIP].x, lm[mp_pose.PoseLandmark.LEFT_HIP].y],
            [lm[mp_pose.PoseLandmark.LEFT_KNEE].x, lm[mp_pose.PoseLandmark.LEFT_KNEE].y],
            [lm[mp_pose.PoseLandmark.LEFT_ANKLE].x, lm[mp_pose.PoseLandmark.LEFT_ANKLE].y]
        )

        right_knee = calculate_angle(
            [lm[mp_pose.PoseLandmark.RIGHT_HIP].x, lm[mp_pose.PoseLandmark.RIGHT_HIP].y],
            [lm[mp_pose.PoseLandmark.RIGHT_KNEE].x, lm[mp_pose.PoseLandmark.RIGHT_KNEE].y],
            [lm[mp_pose.PoseLandmark.RIGHT_ANKLE].x, lm[mp_pose.PoseLandmark.RIGHT_ANKLE].y]
        )

        # ---- AUTO DETECT FRONT / BACK LEG ----
        if left_knee < right_knee:
            front_knee = left_knee
            back_knee = right_knee
        else:
            front_knee = right_knee
            back_knee = left_knee

        # ---- HIPS ----
        left_hip = calculate_angle(
            [lm[mp_pose.PoseLandmark.LEFT_SHOULDER].x, lm[mp_pose.PoseLandmark.LEFT_SHOULDER].y],
            [lm[mp_pose.PoseLandmark.LEFT_HIP].x, lm[mp_pose.PoseLandmark.LEFT_HIP].y],
            [lm[mp_pose.PoseLandmark.LEFT_KNEE].x, lm[mp_pose.PoseLandmark.LEFT_KNEE].y]
        )

        right_hip = calculate_angle(
            [lm[mp_pose.PoseLandmark.RIGHT_SHOULDER].x, lm[mp_pose.PoseLandmark.RIGHT_SHOULDER].y],
            [lm[mp_pose.PoseLandmark.RIGHT_HIP].x, lm[mp_pose.PoseLandmark.RIGHT_HIP].y],
            [lm[mp_pose.PoseLandmark.RIGHT_KNEE].x, lm[mp_pose.PoseLandmark.RIGHT_KNEE].y]
        )

        # ---- ELBOWS ----
        left_elbow = calculate_angle(
            [lm[mp_pose.PoseLandmark.LEFT_SHOULDER].x, lm[mp_pose.PoseLandmark.LEFT_SHOULDER].y],
            [lm[mp_pose.PoseLandmark.LEFT_ELBOW].x, lm[mp_pose.PoseLandmark.LEFT_ELBOW].y],
            [lm[mp_pose.PoseLandmark.LEFT_WRIST].x, lm[mp_pose.PoseLandmark.LEFT_WRIST].y]
        )

        right_elbow = calculate_angle(
            [lm[mp_pose.PoseLandmark.RIGHT_SHOULDER].x, lm[mp_pose.PoseLandmark.RIGHT_SHOULDER].y],
            [lm[mp_pose.PoseLandmark.RIGHT_ELBOW].x, lm[mp_pose.PoseLandmark.RIGHT_ELBOW].y],
            [lm[mp_pose.PoseLandmark.RIGHT_WRIST].x, lm[mp_pose.PoseLandmark.RIGHT_WRIST].y]
        )

        # ---- STORE ----
        pose4_angles["front_knee"].append(front_knee)
        pose4_angles["back_knee"].append(back_knee)
        pose4_angles["left_hip"].append(left_hip)
        pose4_angles["right_hip"].append(right_hip)
        pose4_angles["left_elbow"].append(left_elbow)
        pose4_angles["right_elbow"].append(right_elbow)

        print(f"\nImage: {img_name}")
        print("Front Knee:", int(front_knee))
        print("Back Knee:", int(back_knee))
        print("Left Hip:", int(left_hip))
        print("Right Hip:", int(right_hip))
        print("Left Elbow:", int(left_elbow))
        print("Right Elbow:", int(right_elbow))

# ---- MIN / MAX / AVG ----
pose4_stats = {
    k: {
        "min": int(np.min(v)),
        "max": int(np.max(v)),
        "avg": int(np.mean(v))
    }
    for k, v in pose4_angles.items()
}

print("\nPOSE-4 STATS:", pose4_stats)

# ---- JSON TEMPLATE ----
pose4_template = {
    "pose_name": "Ashwa_Sanchalanasana",
    "angles": {
        "front_knee": [pose4_stats["front_knee"]["min"], pose4_stats["front_knee"]["max"]],
        "back_knee": [pose4_stats["back_knee"]["min"], pose4_stats["back_knee"]["max"]],
        "left_hip": [pose4_stats["left_hip"]["min"], pose4_stats["left_hip"]["max"]],
        "right_hip": [pose4_stats["right_hip"]["min"], pose4_stats["right_hip"]["max"]],
        "left_elbow": [pose4_stats["left_elbow"]["min"], pose4_stats["left_elbow"]["max"]],
        "right_elbow": [pose4_stats["right_elbow"]["min"], pose4_stats["right_elbow"]["max"]]
    }
}

with open("pose4_ashwa_sanchalanasana.json", "w") as f:
    json.dump(pose4_template, f, indent=4)

print("\nPose-4 JSON saved successfully ✅")


Image: 41.jpg
Front Knee: 55
Back Knee: 127
Left Hip: 179
Right Hip: 29
Left Elbow: 170
Right Elbow: 173

Image: 42.jpg
Front Knee: 60
Back Knee: 132
Left Hip: 42
Right Hip: 152
Left Elbow: 162
Right Elbow: 172

Image: 43.jpg
Front Knee: 51
Back Knee: 133
Left Hip: 159
Right Hip: 32
Left Elbow: 178
Right Elbow: 174

POSE-4 STATS: {'front_knee': {'min': 51, 'max': 60, 'avg': 56}, 'back_knee': {'min': 127, 'max': 133, 'avg': 131}, 'left_hip': {'min': 42, 'max': 179, 'avg': 127}, 'right_hip': {'min': 29, 'max': 152, 'avg': 71}, 'left_elbow': {'min': 162, 'max': 178, 'avg': 170}, 'right_elbow': {'min': 172, 'max': 174, 'avg': 173}}

Pose-4 JSON saved successfully ✅


In [14]:
cap = cv2.VideoCapture(0)

SESSION_TIME = 60      # 3 minutes per asana
RELAX_TIME = 10         # 10 seconds rest

asana_files = [
    "pose1_pranamasana.json",
    "pose2_ values.json",
    "pose3_padahastasana.json",
    "pose4_ashwa_sanchalanasana.json"
]

# --------------- ANGLE CALCULATION ---------------
def calculate_angle(a, b, c):
    import numpy as np
    a, b, c = map(np.array, (a, b, c))
    ba = a - b
    bc = c - b
    angle = np.degrees(
        np.arccos(
            np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
        )
    )
    return angle

# --------------- LANDMARK MAP ---------------
def get_angle_value(lm, angle_name):
    P = mp_pose.PoseLandmark

    mapping = {
        "left_elbow": (P.LEFT_SHOULDER, P.LEFT_ELBOW, P.LEFT_WRIST),
        "right_elbow": (P.RIGHT_SHOULDER, P.RIGHT_ELBOW, P.RIGHT_WRIST),
        "left_knee": (P.LEFT_HIP, P.LEFT_KNEE, P.LEFT_ANKLE),
        "right_knee": (P.RIGHT_HIP, P.RIGHT_KNEE, P.RIGHT_ANKLE),
        "left_shoulder": (P.LEFT_ELBOW, P.LEFT_SHOULDER, P.LEFT_HIP),
        "right_shoulder": (P.RIGHT_ELBOW, P.RIGHT_SHOULDER, P.RIGHT_HIP),
        "left_hip": (P.LEFT_SHOULDER, P.LEFT_HIP, P.LEFT_KNEE),
        "right_hip": (P.RIGHT_SHOULDER, P.RIGHT_HIP, P.RIGHT_KNEE),
    }

    if angle_name not in mapping:
        return None

    a, b, c = mapping[angle_name]
    return calculate_angle(
        [lm[a].x, lm[a].y],
        [lm[b].x, lm[b].y],
        [lm[c].x, lm[c].y]
    )

# ---------------- MAIN LOOP ----------------
final_report = {}

for file in asana_files:
    with open(file, "r") as f:
        pose_data = json.load(f)

    pose_name = pose_data["pose_name"]
    pose_ranges = pose_data["angles"]

    print(f"\n🧘 STARTING {pose_name}")

    session_start = time.time()
    pose_start = None
    correct_time = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        current_time = time.time()
        elapsed = current_time - session_start

        if elapsed >= SESSION_TIME:
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(frame_rgb)

        pose_correct = True
        feedback = []

        if results.pose_landmarks:
            lm = results.pose_landmarks.landmark

            for angle_name, (min_v, max_v) in pose_ranges.items():
                angle_val = get_angle_value(lm, angle_name)

                if angle_val is None or not (min_v <= angle_val <= max_v):
                    pose_correct = False
                    feedback.append(f"Adjust {angle_name.replace('_',' ')}")

        # -------- CUMULATIVE TIMER --------
        if pose_correct and results.pose_landmarks:
            if pose_start is None:
                pose_start = current_time
        else:
            if pose_start:
                correct_time += current_time - pose_start
                pose_start = None

        # -------- DISPLAY --------
        cv2.putText(frame, pose_name, (20, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,0), 2)

        cv2.putText(frame, f"Time: {int(elapsed)}/180 sec", (20, 75),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

        cv2.putText(frame, f"Correct: {int(correct_time)} sec", (20, 110),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

        y = 150
        for msg in feedback:
            cv2.putText(frame, msg, (20, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)
            y += 30

        cv2.imshow("Surya Namaskar Trainer", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            cap.release()
            cv2.destroyAllWindows()
            exit()

    if pose_start:
        correct_time += time.time() - pose_start

    accuracy = round((correct_time / SESSION_TIME) * 100, 2)
    final_report[pose_name] = accuracy

    # -------- RELAXATION TIME --------
    relax_start = time.time()
    while time.time() - relax_start < RELAX_TIME:
        ret, frame = cap.read()
        if not ret:
            break

        remaining = RELAX_TIME - int(time.time() - relax_start)
        cv2.putText(frame, "RELAX", (180, 200),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0,255,255), 3)
        cv2.putText(frame, f"Next in {remaining}s", (180, 250),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)

        cv2.imshow("Surya Namaskar Trainer", frame)
        cv2.waitKey(1)

# ---------------- FINAL REPORT ----------------
print("\n===== FINAL ACCURACY REPORT =====")
for k, v in final_report.items():
    print(f"{k}: {v}%")

cap.release()
cv2.destroyAllWindows()


🧘 STARTING Pranamasana

🧘 STARTING Hasta_Uttanasana

🧘 STARTING Padahastasana

🧘 STARTING Ashwa_Sanchalanasana

===== FINAL ACCURACY REPORT =====
Pranamasana: 19.04%
Hasta_Uttanasana: 2.0%
Padahastasana: 0.0%
Ashwa_Sanchalanasana: 0.0%


In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import json
import os

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True)

# 👉 GIVE YOUR IMAGE PATH HERE
image_path = "C:\SDP PROJECT\static\Pose7-P.jpeg"

# -------- READ IMAGE --------
image = cv2.imread(image_path)

if image is None:
    print("❌ Image not found")
    exit()

image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
results = pose.process(image_rgb)

if not results.pose_landmarks:
    print("❌ No pose detected")
    exit()

lm = results.pose_landmarks.landmark

# -------- ANGLE FUNCTION --------
def calc(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b
    angle = np.degrees(np.arccos(
        np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    ))
    return int(angle)

# -------- CALCULATE ANGLES --------
angles = {}

angles["left_elbow"] = calc([lm[11].x, lm[11].y],
                            [lm[13].x, lm[13].y],
                            [lm[15].x, lm[15].y])

angles["right_elbow"] = calc([lm[12].x, lm[12].y],
                             [lm[14].x, lm[14].y],
                             [lm[16].x, lm[16].y])

angles["left_knee"] = calc([lm[23].x, lm[23].y],
                           [lm[25].x, lm[25].y],
                           [lm[27].x, lm[27].y])

angles["right_knee"] = calc([lm[24].x, lm[24].y],
                            [lm[26].x, lm[26].y],
                            [lm[28].x, lm[28].y])

angles["left_hip"] = calc([lm[11].x, lm[11].y],
                          [lm[23].x, lm[23].y],
                          [lm[25].x, lm[25].y])

angles["right_hip"] = calc([lm[12].x, lm[12].y],
                           [lm[24].x, lm[24].y],
                           [lm[26].x, lm[26].y])

# -------- CREATE ±5 RANGE --------
angle_ranges = {}

for key, value in angles.items():
    angle_ranges[key] = [value - 5, value + 5]

# -------- JSON FORMAT --------
pose_json = {
    "pose_name": os.path.basename(image_path).split('.')[0],
    "angles": angle_ranges
}

# -------- SAVE JSON --------
json_name = os.path.basename(image_path).split('.')[0] + ".json"

with open(json_name, "w") as f:
    json.dump(pose_json, f, indent=4)

print(f"✅ JSON saved as: {json_name}")

❌ No pose detected


AttributeError: 'NoneType' object has no attribute 'landmark'

: 